# 05 - Model zona kedaluwarsa 4 KELAS

Melatih YOLOv8 membedakan **`date` / `code` / `due` / `prod`**, menggantikan model
1 kelas (`expired_date_zone`).

**Kenapa:** jawaban salah `chees` berasal dari parser yang membaca nomor produksi
`210526` sebagai tanggal. Perbaikan dari sisi parser sudah dicoba 20-11 dan **gagal** -
teks OCR-nya tidak memuat label "Kode Produksi" sama sekali. Model 4 kelas menolak
wilayah kode produksi **sebelum OCR**, jadi labelnya tidak perlu terbaca.

**Tahan harapan:** batas atas ~**+6 poin** (1 berkas dari 16). Jalur zona sudah terukur
sebagai optimasi **latensi**, bukan akurasi (81% dengan maupun tanpa, 02-10).

Mengikuti pola notebook 01/03/04: **semua hasil ditulis langsung ke Drive**, supaya
tidak hilang saat sesi Colab terputus dan bisa dibaca tanpa menunggu siapa pun.


In [ ]:
# Sel 1 - GPU dan Drive
# Penjaga GPU: latihan k-fold uang v3 pernah diam-diam jalan di CPU
# (1 epoch = 965 detik) karena !nvidia-smi hanya MENCETAK, tidak menghentikan.
import torch
assert torch.cuda.is_available(), (
    'Runtime TANPA GPU. Runtime > Change runtime type > T4 GPU, lalu jalankan ulang.')
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Sel 2 - pustaka
%pip -q install ultralytics roboflow
import ultralytics
print('ultralytics', ultralytics.__version__)


In [ ]:
# Sel 3 - folder kerja di Drive (pola notebook 01/03/04)
from pathlib import Path
DRIVE_DIR  = Path('/content/drive/MyDrive/sayit')
RUNS_DIR   = DRIVE_DIR / 'runs'
MODELS_DIR = DRIVE_DIR / 'models'
HASIL_DIR  = DRIVE_DIR / 'hasil'      # ringkasan metrik, agar bisa dibaca tanpa Colab
for d in (RUNS_DIR, MODELS_DIR, HASIL_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Folder kerja:', DRIVE_DIR)


In [ ]:
# Sel 4 - konfigurasi.
#
# Kunci dibaca dari Colab Secrets (ikon kunci di bilah kiri), BUKAN ditulis di sel:
# notebook ini ikut di-commit, dan kunci yang tertulis di sel akan ikut masuk Git.
# Nama secret: ROBOFLOW_API_KEY, saklar 'Notebook access' MENYALA.
# WAJIB kunci PRIVATE - publishable key berawalan rf_ ditolak 401.
PESAN = ('Secret ROBOFLOW_API_KEY tidak terbaca. Buka ikon kunci di bilah kiri, '
         'buat secret bernama persis ROBOFLOW_API_KEY, isi dengan Private API Key, '
         'lalu nyalakan saklar Notebook access.')
try:
    from google.colab import userdata
    RF_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception as e:
    raise SystemExit(f'{PESAN} Pesan aslinya: {e}')
assert RF_API_KEY, PESAN

WORKSPACE   = 'esigelec-rniez'   # 700 gambar, 4 kelas, CC BY 4.0, split lengkap
PROJECT     = 'expiry'
VERSI       = 1                  # diperiksa 20-11: versi 1 satu-satunya yang ada
EPOCHS      = 80
IMGSZ       = 640                # samakan dengan vision.expiry.imgsz
RUN_NAME    = 'zone_4kelas_v1'
KELAS_WAJIB = {'date', 'code', 'due', 'prod'}
print('kunci terbaca,', len(RF_API_KEY), 'karakter')


In [ ]:
# Sel 5 - unduh dataset
from roboflow import Roboflow
rf = Roboflow(api_key=RF_API_KEY)
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSI).download('yolov8')
data_yaml = Path(ds.location) / 'data.yaml'
print('diunduh ke:', ds.location)


In [ ]:
# Sel 6 - PERIKSA DATASET SEBELUM MELATIH.
# Dua jebakan yang sudah merusak latihan di proyek ini:
#   1. nama kelas tidak sesuai harapan (dataset Roboflow berlabel tertukar, 27-09)
#   2. gambar TANPA label di split train - dalam YOLO itu berarti 'tidak ada objek
#      di gambar ini', dan itu merusak k-fold uang pertama (07-10)
import collections
import yaml
d = yaml.safe_load(data_yaml.read_text())
nama = {i: n for i, n in enumerate(d['names'])}
print('kelas:', nama)
kurang = KELAS_WAJIB - {n.lower() for n in nama.values()}
assert not kurang, f'Kelas yang diharapkan tidak ada: {kurang}. Periksa datasetnya.'

for split in ('train', 'valid', 'test'):
    img = Path(ds.location, split, 'images')
    if not img.is_dir():
        print(f'{split:6} TIDAK ADA')
        continue
    lab = Path(ds.location, split, 'labels')
    gambar = sorted(p for p in img.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    kosong, hitung = 0, collections.Counter()
    for g in gambar:
        t = lab / (g.stem + '.txt')
        baris = [b for b in t.read_text().splitlines() if b.strip()] if t.exists() else []
        if not baris:
            kosong += 1
        for b in baris:
            hitung[nama.get(int(b.split()[0]), '?')] += 1
    print(f'{split:6} {len(gambar):5} gambar, {kosong:4} TANPA label, kotak: {dict(hitung)}')
print('Kalau jumlah TANPA label di train besar, HENTIKAN dan periksa dulu.')


In [ ]:
# Sel 7 - latih, LANGSUNG KE DRIVE.
#
# `project` harus jalur ABSOLUT ke Drive (pola notebook 03). `project` relatif
# ditambahkan di belakang settings['runs_dir'] ultralytics, dan di Colab itu sudah
# bernilai /content/runs/detect - sehingga project='runs/detect' menghasilkan
#   /content/runs/detect/runs/detect/<nama>/weights/best.pt
# Terbukti 20-11: Sel evaluasi gagal FileNotFoundError padahal latihannya berhasil.
from ultralytics import YOLO
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / 'weights' / 'best.pt'

model = YOLO('yolov8n.pt')
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, seed=0, deterministic=True,
            project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True, plots=True)

print('save_dir sesungguhnya :', model.trainer.save_dir)
print('bobot ada             :', BEST_PT.exists())
assert BEST_PT.exists(), f'Latihan selesai tetapi {BEST_PT} tidak ada.'


In [ ]:
# Sel 8 - evaluasi split TEST, dan TULIS HASILNYA KE DRIVE
import json
from datetime import datetime
terbaik = YOLO(str(BEST_PT))
m = terbaik.val(data=str(data_yaml), split='test', imgsz=IMGSZ,
                project=str(RUNS_DIR), name=RUN_NAME + '_test', exist_ok=True)

per_kelas = {}
for i, n in nama.items():
    try:
        per_kelas[n] = round(float(m.box.ap50[i]), 4)
    except (IndexError, TypeError):
        pass

ringkas = {
    'run': RUN_NAME,
    'waktu': datetime.now().isoformat(timespec='seconds'),
    'dataset': f'{WORKSPACE}/{PROJECT} v{VERSI}',
    'epochs': EPOCHS, 'imgsz': IMGSZ,
    'mAP50': round(float(m.box.map50), 4),
    'mAP50_95': round(float(m.box.map), 4),
    'ap50_per_kelas': per_kelas,
    'bobot': str(BEST_PT),
}
berkas = HASIL_DIR / f'{RUN_NAME}_metrik.json'
berkas.write_text(json.dumps(ringkas, indent=2), encoding='utf-8')
print(json.dumps(ringkas, indent=2))
print('Ringkasan ditulis ke', berkas)
print('Yang menentukan untuk aplikasi: AP kelas date dan code.')
print('code penting karena gunanya MENOLAK, bukan membaca.')


In [ ]:
# Sel 9 - salin bobot ke Drive dengan nama bertanggal (pola notebook 03)
import shutil
mb = BEST_PT.stat().st_size / 1e6
# Ultralytics membuang state optimizer HANYA saat latihan berakhir normal.
# ~6 MB = selesai; ~32 MB = terputus (pelajaran k-fold uang v3, 09-11).
print(f'{BEST_PT} = {mb:.1f} MB ->',
      'selesai normal' if mb < 15 else 'TERPUTUS, jangan dipakai')

stamp = datetime.now().strftime('%Y%m%d_%H%M')
dst = MODELS_DIR / f'best_zone_4kelas_{RUN_NAME}_{stamp}.pt'
shutil.copy(BEST_PT, dst)
print('Bobot disalin ke', dst)
print()
print('Di laptop:')
print(r'   copy <berkas>.pt models\best_zone_4kelas.pt')
print('JANGAN menimpa models/best_zone.pt - itu bukti angka 81% yang sekarang.')


## Sesudah ini

Semua hasil sudah ada di Drive:

| isi | tempat |
|---|---|
| bobot + grafik latihan | `MyDrive/sayit/runs/zone_4kelas_v1/` |
| bobot bertanggal | `MyDrive/sayit/models/best_zone_4kelas_*.pt` |
| **ringkasan metrik** | `MyDrive/sayit/hasil/zone_4kelas_v1_metrik.json` |

JSON metrik itu yang membuat hasilnya bisa dibaca tanpa membuka Colab.

Lalu di laptop, **ukur sebelum menyimpulkan**:

```
python scripts\ukur_kedaluwarsa.py
```

Bandingkan **benar / salah / diam** terhadap dasar **13 / 2 / 1 = 81%** - bukan satu
angka akurasi. Arah kesalahan di modul ini sudah tiga kali terbukti lebih penting
daripada jumlahnya.

mAP yang tinggi di split test Roboflow **tidak** menjamin perbaikan di 16 kemasan kita:
split itu sesumber dengan data latihnya, sedangkan kemasan kita foto sendiri.
